# Political Debater

We will build two agents in langchain. Each will argue the claim. A final LLM will summarize the debate in a neutral way.

## The Code

In [15]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
load_dotenv(override=True)

#MODEL = 'gpt-4.1-mini'
MODEL = 'gpt-6-astra'
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")


In [16]:
from typing import Any
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

In [17]:
def run_agent(agent, messages):

    full_response = ""

    for token, metadata in agent.stream(
        {"messages": messages},
        stream_mode="messages",
    ):

        if token.content:
            print(token.content, end="", flush=True)
            full_response += token.content

    print()

    return full_response

In [18]:
# ============================================================
# Configuration
# ============================================================
DEFAULT_ROUNDS = 3

In [19]:
# ============================================================
# LLM
# ============================================================

model = ChatOpenAI(
    base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, 
    model=MODEL,
    temperature=0.2,
)

In [20]:
# ============================================================
# System prompts
# ============================================================

ADVOCATE_PROMPT = """
You are Agent A: the ADVOCATE in a structured debate.

Your job is to argue FOR the claim. Use layman language.

Responsibilities:

- Construct the strongest reasonable argument supporting the claim.
- Respond directly to the critic's arguments.
- Identify weaknesses or unsupported assumptions in opposing arguments.
- Use evidence when appropriate (show links to sources).
- Distinguish factual evidence from inference and opinion.
- Do not invent evidence, statistics, studies, quotations, or sources.
- Use the available evidence tool when factual evidence would materially
  strengthen or verify your argument.
- Do not change sides.
- Do not merely repeat arguments you have already made.
- Focus on unresolved issues from previous rounds.

Keep your argument concise and structured.
"""


CRITIC_PROMPT = """
You are Agent B: the CRITIC in a structured debate.

Your job is to argue AGAINST the claim. Use layman language.

Responsibilities:

- Construct the strongest reasonable argument opposing the claim.
- Respond directly to the advocate's arguments.
- Identify weaknesses, assumptions, and logical problems.
- Use evidence when appropriate (show links to sources).
- Distinguish factual evidence from inference and opinion.
- Do not invent evidence, statistics, studies, quotations, or sources.
- Use the available evidence tool when factual evidence would materially
  strengthen or verify your argument.
- Do not change sides.
- Do not merely repeat arguments you have already made.
- Focus on unresolved issues from previous rounds.

Keep your argument concise and structured.
"""

# ============================================================
# Transcript summariser
# ============================================================

SUMMARISER_PROMPT = """
You are a neutral debate summariser.

Your job is to summarise a debate between an Advocate and a Critic.
You respond in neat markdown format.

Rules:

- Remain neutral.
- Do not decide who won.
- Do not introduce new arguments or facts.
- Only summarise information contained in the transcript.
- Clearly distinguish the Advocate's arguments from the Critic's.
- Identify the major areas of disagreement.
- Identify any areas where the two sides agree.
- Include important rebuttals and counterarguments.
- Avoid unnecessary repetition.

Produce the summary using these sections:

1. Claim
2. Advocate's Main Arguments
3. Critic's Main Arguments
4. Key Rebuttals
5. Areas of Agreement
6. Main Unresolved Disagreements
7. Overall Neutral Summary
"""


POLICY_TO_CLAIM_PROMPT = """
You are an agent that converts policy statements into debate claims.

Your job is to take a given policy and formulate a clear, concise, and debatable claim that can be argued for or against.

Rules:

- The claim must be specific and actionable.
- The claim must be debatable (i.e., it should be possible to argue both for and against it).
- Avoid vague or overly broad statements.
- Do not include supporting arguments or evidence in the claim itself.
- Keep the claim concise and clear.

Example:

Policy: "The government should increase funding for renewable energy research."
Claim: "The government should significantly increase funding for renewable energy research."

Produce a claim based on the following policy:

Policy:
{policy}

Claim:
"""

In [21]:
# ============================================================
# Create agents
# ============================================================

advocate = create_agent(
    model=model,
    tools=[],
    system_prompt=ADVOCATE_PROMPT,
)

critic = create_agent(
    model=model,
    tools=[],
    system_prompt=CRITIC_PROMPT,
)

summariser = create_agent(
    model=model,
    tools=[],
    system_prompt=SUMMARISER_PROMPT,
)

policy_to_claim = create_agent(
    model=model,
    tools=[],
    system_prompt=POLICY_TO_CLAIM_PROMPT,
)


In [22]:
# ============================================================
# Transcript helpers
# ============================================================

def format_transcript(transcript: list[dict[str, str]]) -> str:
    """
    Convert the debate transcript into text suitable for
    providing back to the agents.
    """

    if not transcript:
        return "No arguments have been made yet."

    sections = []

    for item in transcript:
        sections.append(
            f"{item['speaker']}:\n"
            f"{item['content']}"
        )

    return "\n\n".join(sections)

In [23]:

# ============================================================
# Streaming agent runner
# ============================================================

def run_agent(
    agent: Any,
    prompt: str,
) -> str:
    """
    Run an agent and stream its final response.

    Tool execution is handled automatically by the LangChain agent.

    Returns:
        The complete textual response so it can be added to the
        debate transcript.
    """

    full_response = ""

    for token, metadata in agent.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ]
        },
        stream_mode="messages",
    ):

        # ----------------------------------------------------
        # Only print ordinary textual model output.
        #
        # Tool-call chunks may also pass through the message
        # stream, so ignore chunks without normal text.
        # ----------------------------------------------------

        content = getattr(token, "content", None)

        if isinstance(content, str) and content:
            print(content, end="", flush=True)
            full_response += content

    print()

    return full_response.strip()


In [24]:

# ============================================================
# Debate
# ============================================================

def debate(
    claim: str,
    rounds: int = DEFAULT_ROUNDS,
) -> list[dict[str, str]]:
    """
    Conduct a debate between the Advocate and Critic.

    Flow:

        Advocate opening

        Critic response
        Advocate response

        Critic response
        Advocate response

        ...

    Both agents receive the complete debate transcript before
    producing each new argument.
    """

    transcript: list[dict[str, str]] = []

    # ========================================================
    # Display claim
    # ========================================================

    print()
    print("=" * 70)
    print("CLAIM")
    print("=" * 70)
    print(claim)

    # ========================================================
    # Advocate opening statement
    # ========================================================

    print()
    print("=" * 70)
    print("ADVOCATE — OPENING STATEMENT")
    print("=" * 70)

    prompt = f"""
CLAIM:

{claim}


You are opening the debate.

Present the strongest reasonable argument FOR the claim.

You may use the evidence search tool if factual evidence would
help establish your argument.
"""

    response = run_agent(
        advocate,
        prompt,
    )

    transcript.append(
        {
            "speaker": "Advocate",
            "content": response,
        }
    )

    # ========================================================
    # Debate rounds
    # ========================================================

    for round_number in range(1, rounds + 1):

        # ----------------------------------------------------
        # Critic
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(f"ROUND {round_number} — CRITIC")
        print("=" * 70)

        history = format_transcript(transcript)

        critic_prompt = f"""
CLAIM:

{claim}


DEBATE SO FAR:

{history}


You are the CRITIC.

Respond to the arguments made so far and argue AGAINST the claim.

Requirements:

1. Address the advocate's strongest argument.
2. Identify unsupported assumptions where appropriate.
3. Provide counterarguments.
4. Use the evidence search tool if factual verification would
   materially improve your argument.
5. Do not simply repeat arguments already made.
"""

        response = run_agent(
            critic,
            critic_prompt,
        )

        transcript.append(
            {
                "speaker": "Critic",
                "content": response,
            }
        )

        # ----------------------------------------------------
        # Advocate
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(f"ROUND {round_number} — ADVOCATE")
        print("=" * 70)

        history = format_transcript(transcript)

        advocate_prompt = f"""
CLAIM:

{claim}


DEBATE SO FAR:

{history}


You are the ADVOCATE.

Respond to the critic's arguments and continue arguing FOR
the claim.

Requirements:

1. Address the critic's strongest objection.
2. Defend or refine your previous argument.
3. Identify weaknesses in the critic's reasoning.
4. Use the evidence search tool if factual verification would
   materially improve your argument.
5. Do not simply repeat arguments already made.
"""

        response = run_agent(
            advocate,
            advocate_prompt,
        )

        transcript.append(
            {
                "speaker": "Advocate",
                "content": response,
            }
        )

    # ========================================================
    # Finished
    # ========================================================

    print()
    print("=" * 70)
    print("DEBATE COMPLETE")
    print("=" * 70)

    return transcript




In [25]:
# ============================================================
# Display transcript
# ============================================================

def transcript_to_markdown(
    transcript: list[dict[str, str]],
) -> None:
    md = ""
    for index, item in enumerate(transcript, start=1):
        md += f"## [{index}] {item['speaker'].upper()}\n\n{item['content']}\n\n"

    return md

# ============================================================
# Summarise debate
# ============================================================

def summarise_debate(
    claim: str,
    transcript: list[dict[str, str]],
) -> str:
    """
    Summarise the complete debate transcript.

    Args:
        claim:
            The original claim being debated.

        transcript:
            Complete debate transcript containing Advocate
            and Critic responses.

    Returns:
        Neutral summary of the entire debate.
    """

    full_transcript = format_transcript(transcript)

    prompt = f"""
ORIGINAL CLAIM:

{claim}


COMPLETE DEBATE TRANSCRIPT:

{full_transcript}


Summarise the entire debate.

Your summary must represent both sides fairly and must only use
information contained in the transcript.
"""

    summary = run_agent(
        summariser,
        prompt,
    )

    return summary

# ============================================================
# Convert policy to claim
# ============================================================

def convert_policy_to_claim(
    policy: str,
) -> str:
    """
    Convert a policy statement into a clear, concise, and debatable claim.

    Args:
        policy:
            The policy statement to be converted into a claim.

    Returns:
        A debatable claim derived from the policy statement.
    """

    prompt = f"""
Policy:
{policy}

Claim:
"""

    claim = run_agent(
        policy_to_claim,
        prompt,
    )

    return claim


## Crawler

In [26]:
from scraper import fetch_website_links, fetch_website_contents  # Written by Ed Donner


# Pauline Hanson's One Nation Policies

Pauline Hanson is a far right wing populist politician. https://en.wikipedia.org/wiki/Pauline_Hanson

We will debate each of her political policies, one by one. 

A neutral summary of each debate topic will be given at the end of each full transcript.

In [27]:
policy_urls1 = [
    "https://www.onenation.org.au/oil-and-gas",
    "https://www.onenation.org.au/famly-tax-policy",
    "https://www.onenation.org.au/slash-waste",
    "https://www.onenation.org.au/free-speech-policy",
    "https://www.onenation.org.au/reduce-cost-of-living",
    "https://www.onenation.org.au/citizen-initiated-referenda",
    "https://www.onenation.org.au/climate",
    "https://www.onenation.org.au/family",
    "https://www.onenation.org.au/farming",
    "https://www.onenation.org.au/foreign-ownership",
    "https://www.onenation.org.au/education",
    "https://www.onenation.org.au/immigration",
    "https://www.onenation.org.au/cannabis",
    "https://www.onenation.org.au/rorts",
    "https://www.onenation.org.au/tax",
    "https://www.onenation.org.au/firearms",
    "https://www.onenation.org.au/health",
    "https://www.onenation.org.au/covid",
    "https://www.onenation.org.au/pro-life",
    "https://www.onenation.org.au/fisheries",
    "https://www.onenation.org.au/forestry",
    "https://www.onenation.org.au/real-impacts-of-net-zero-australia",
    "https://www.onenation.org.au/net-zero-destroying-australia",
    "https://www.onenation.org.au/embracing-australian-coal-reliable-energy-future",
    "https://www.onenation.org.au/embracing-nuclear-energy-hydro-australia-future",
    "https://www.onenation.org.au/lowering-prices-households-business-industry-australia"
]

In [28]:
from pathlib import Path
import re
from time import time
    
OUTPUT_DIR = Path(f"ai-debates-{MODEL}-{int(time())}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def slugify(text: str, max_length: int = 80) -> str:
    """Convert text into a safe filename."""
    slug = text.lower().strip()
    slug = re.sub(r"[^\w\s-]", "", slug)
    slug = re.sub(r"[-\s]+", "-", slug)
    return slug[:max_length].strip("-")

for index, policy_url in enumerate(policy_urls1, start=1):

    try:
        # --------------------------------------------------
        # Fetch policy
        # --------------------------------------------------

        policy_content = fetch_website_contents(policy_url)

        # --------------------------------------------------
        # Convert policy into a debatable claim
        # --------------------------------------------------

        claim = convert_policy_to_claim(policy_content)

        print(f"\n{'=' * 80}")
        print(f"Debate {index}: {claim}")
        print(f"{'=' * 80}")

        # --------------------------------------------------
        # Run debate
        # --------------------------------------------------

        transcript = debate(
            claim=claim,
            rounds=3,
        )

        # --------------------------------------------------
        # Summarise debate
        # --------------------------------------------------

        summary = summarise_debate(
            claim=claim,
            transcript=transcript,
        )

        # --------------------------------------------------
        # Write debate to Markdown
        # --------------------------------------------------

        filename = f"{index:03d}-{slugify(claim)}.md"
        output_path = OUTPUT_DIR / filename

        markdown = f"""# A Multi Agent AI Policy Debate
AI can make mistakes. Always critically evaluate the content of the debate.

## Claim

{claim}

## Source

{policy_url}

## Debate

{transcript_to_markdown(transcript)}

## Summary

{summary}
"""

        output_path.write_text(
            markdown,
            encoding="utf-8",
        )

        print(f"Saved: {output_path}")

    except Exception as exc:
        print(f"ERROR processing {policy_url}: {exc}")

The Australian government should fund 30% of oil and gas exploration costs in Commonwealth waters in exchange for public equity stakes of up to 30% in resulting production licences.

Debate 1: The Australian government should fund 30% of oil and gas exploration costs in Commonwealth waters in exchange for public equity stakes of up to 30% in resulting production licences.

CLAIM
The Australian government should fund 30% of oil and gas exploration costs in Commonwealth waters in exchange for public equity stakes of up to 30% in resulting production licences.

ADVOCATE — OPENING STATEMENT
## Opening argument: public risk should buy public ownership

Australia should fund 30% of offshore oil and gas exploration costs **in return for a commercially fair public stake**, because that gives Australians a direct share of successful projects—not just the risks and costs associated with developing them.

**1. Australians should earn an owner’s return from a public resource.**  
Taxes remain esse